# 01 — Getting started with `VectorCollection`

This notebook walks through the `vectors-db` public API: opening a collection, adding documents, searching, and switching between `FLAT` and `HNSW` indexes.

The kernel is JJava running on JDK 25 with the Panama Vector API enabled, so every distance kernel behind the scenes is SIMD-accelerated.

**Prerequisite:** run `./gradlew build -x :docs:build -x test` on the host once before launching this notebook.

In [ ]:
// Pick up all module jars. The glob resolves to the jars produced by the
// last `./gradlew build`; re-run the host build whenever library source changes.
%jars /home/jovyan/work/vectors/vectors-core/build/libs/vectors-core-*.jar
%jars /home/jovyan/work/vectors/vectors-storage/build/libs/vectors-storage-*.jar
%jars /home/jovyan/work/vectors/vectors-quantization/build/libs/vectors-quantization-*.jar
%jars /home/jovyan/work/vectors/vectors-hnsw/build/libs/vectors-hnsw-*.jar
%jars /home/jovyan/work/vectors/vectors-db/build/libs/vectors-db-*.jar

In [ ]:
import com.integrallis.vectors.core.SimilarityFunction;
import com.integrallis.vectors.db.Document;
import com.integrallis.vectors.db.IndexType;
import com.integrallis.vectors.db.SearchRequest;
import com.integrallis.vectors.db.SearchResult;
import com.integrallis.vectors.db.VectorCollection;
import java.util.Random;

final int DIMENSION = 32;
final Random rnd = new Random(42L);

float[] unit(int dim) {
    float[] v = new float[dim];
    float norm = 0f;
    for (int i = 0; i < dim; i++) { v[i] = (float) rnd.nextGaussian(); norm += v[i]*v[i]; }
    norm = (float) Math.sqrt(norm);
    if (norm > 0f) for (int i = 0; i < dim; i++) v[i] /= norm;
    return v;
}

## FLAT (exact) collection

`FLAT` is a brute-force scan. Ideal when you have ≤100k vectors or when you want an unambiguous recall=1.0 baseline to compare against approximate indexes.

In [ ]:
VectorCollection flat = VectorCollection.builder()
    .dimension(DIMENSION)
    .metric(SimilarityFunction.COSINE)
    .indexType(IndexType.FLAT)
    .autoCommitThreshold(1)
    .build();

for (int i = 0; i < 500; i++) {
    flat.add(Document.of("doc-" + i, unit(DIMENSION), "document " + i));
}
flat.commit();

float[] query = unit(DIMENSION);
SearchResult r = flat.search(SearchRequest.builder(query, 5).includeText(true).build());
for (SearchResult.Hit h : r.hits()) {
    System.out.printf("  %-8s  score=%.4f  text=%s%n", h.document().id(), h.score(), h.document().text());
}
System.out.println("collection size: " + flat.size());

## HNSW (approximate) collection

`HNSW` is a graph-based ANN index. For the same 500 documents it will match `FLAT` nearly perfectly at microsecond latencies, and the gap widens dramatically at a million+ vectors. The `hnswM` and `hnswEfConstruction` tuning knobs govern the build-time recall/latency tradeoff.

In [ ]:
VectorCollection hnsw = VectorCollection.builder()
    .dimension(DIMENSION)
    .metric(SimilarityFunction.COSINE)
    .indexType(IndexType.HNSW)
    .hnswM(16)
    .hnswEfConstruction(100)
    .autoCommitThreshold(1)
    .build();

for (Document d : flat.documents()) hnsw.add(d);
hnsw.commit();

SearchResult rHnsw = hnsw.search(SearchRequest.builder(query, 5).build());
for (SearchResult.Hit h : rHnsw.hits()) {
    System.out.printf("  %-8s  score=%.4f%n", h.document().id(), h.score());
}

flat.close();
hnsw.close();

## Next up

- **02 — Quantization tour:** SQ8, PQ, BQ compression-vs-recall on a larger synthetic dataset.
- **03 — Spring AI:** drop `JavaVectorsVectorStore` into a RAG pipeline.
- **04 — LangChain4j:** `JavaVectorsEmbeddingStore` end-to-end.